# Setup

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu transformers accelerate langchain langchain-community langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os, warnings
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

warnings.filterwarnings('ignore')

device = 0 if torch.cuda.is_available() else -1
print(f"{'GPU' if device == 0 else 'CPU'}")

FAISS_PATH_256 = "faiss_256"
FAISS_PATH_512 = "faiss_512"


/tmp/ipykernel_1341/3760674100.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


GPU


# Ingestão

Realizei inicialmente um teste com apenas 500 artigos, porém as perguntas que fazia frequentemente não tinham um artigo correspondente indexado, o que tornava difícil avaliar se o RAG funcionava de fato ou se o modelo simplesmente não tinha o que recuperar. Decidi para a solução final utilizar 3000 artigos aumentando a cobertura, mas sem estourar a memória do Colab.


In [ ]:
dataset = load_dataset("TucanoBR/wikipedia-PT", split="train[:3000]")

colunas = dataset.column_names
col_texto = 'text' if 'text' in colunas else colunas[0]

documents = [
    Document(page_content=row[col_texto])
    for row in dataset
    if row[col_texto] and len(row[col_texto].strip()) > 0
]

print(f"Artigos carregados: {len(documents)}")
print(f"\nExemplo do primeiro artigo:")
print(documents[0].page_content[:300])


README.md:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/260M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/261M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/246M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1103446 [00:00<?, ? examples/s]

Artigos carregados: 3000

Exemplo do primeiro artigo:
Astronomia é uma ciência natural que estuda corpos celestes (como estrelas, planetas, cometas, nebulosas, aglomerados de estrelas, galáxias) e fenômenos que se originam fora da atmosfera da Terra (como a radiação cósmica de fundo em micro-ondas). Preocupada com a evolução, a física e a química de ob


# Chunking e Tokens

O **RecursiveCharacterTextSplitter** padrão divide por caracteres e caracteres é muito menos que tokens. O modelo de *embedding* vai ver *chunks* acaba vendo tamanhos bem diferentes do esperado. A solução é usar o *tokenizer* do próprio modelo de *embedding* para medir o tamanho real dos *chunks*.

Também testei dois tamanhos para ver o impacto na recuperação: chunks menores tendem a ser mais precisos mas perdem contexto, já chunks maiores carregam mais informação mas podem trazer ruído junto.

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)

splitter_256 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=256, chunk_overlap=50)
splitter_512 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=384, chunk_overlap=100) # limitado a 384 (teto do modelo)

chunks_256 = splitter_256.split_documents(documents)
chunks_512 = splitter_512.split_documents(documents)

print(f"Chunks de 256 tokens: {len(chunks_256):,}")
print(f"Chunks de 384 tokens: {len(chunks_512):,}")
print(f"\nUm chunk de 256 tokens tem em média {sum(len(c.page_content) for c in chunks_256[:100]) // 100} caracteres.")


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (690 > 512). Running this sequence through the model will result in indexing errors


Chunks de 256 tokens: 31,341
Chunks de 384 tokens: 20,961

Um chunk de 256 tokens tem em média 731 caracteres.


# Embeddings e Indexação FAISS

O FAISS indexa vetores numéricos de cada chunk trasformado pelo embedding para que, dada uma pergunta, consigamos encontrar rapidamente os chunks mais próximos semanticamente.

Uma escolha importante aqui foi trocar o **all-MiniLM-L6-v2** pelo **paraphrase-multilingual-MiniLM-L12-v2**, após testes. O primeiro foi treinado quase exclusivamente em inglês, o que interferiu de maneira significativa nos resultados.

Os índices são salvos em disco após a primeira criação para agilizar testes subsequentes.


In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 128 if torch.cuda.is_available() else 32}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
if os.path.exists(FAISS_PATH_256):
    print("Carregando índice 256.")
    vectorstore_256 = FAISS.load_local(FAISS_PATH_256, embeddings_model,
                                        allow_dangerous_deserialization=True)
else:
    print("Criando índice 256.")
    vectorstore_256 = FAISS.from_documents(chunks_256, embeddings_model)
    vectorstore_256.save_local(FAISS_PATH_256)

print("Concluído.")


Carregando índice 256.
Concluído.


In [ ]:
if os.path.exists(FAISS_PATH_512):
    print("Carregando índice 384.")
    vectorstore_512 = FAISS.load_local(FAISS_PATH_512, embeddings_model, allow_dangerous_deserialization=True)
else:
    print("Criando índice 384.")
    vectorstore_512 = FAISS.from_documents(chunks_512, embeddings_model)
    vectorstore_512.save_local(FAISS_PATH_512)

print("Concluído.")


Carregando índice 384.
Concluído.


In [ ]:
retriever_256 = vectorstore_256.as_retriever(search_kwargs={"k": 3})
retriever_512 = vectorstore_512.as_retriever(search_kwargs={"k": 3})


# Modelo Gerador

Para o gerador, usei o TinyLlama, ele tem alguimas limitações: foi treinado majoritariamente em inglês e não segue instruções de restrições de forma confiável.

In [ ]:
print("Carregando TinyLlama.")

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=device,
    max_new_tokens=80,
    max_length=None,
    temperature=0.1,
    do_sample=True,
)
llm = HuggingFacePipeline(pipeline=pipe)

print("Concluído.")


Carregando TinyLlama.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Concluído.


## Comparações

O prompt está em inglês porque o TinyLlama responde com mais coerência nesse idioma e em testes anteriores ele não consegui entender o que estava sendo pedido.

A instrução ***"... say 'I don't know'"*** testa se o modelo consegue "entender" quando o contexto não é suficiente, algo que ele mostyrou ter dificuldades.

In [ ]:
def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt_closed = PromptTemplate.from_template(
    "Answer concisely in one or two sentences: {query}"
)

prompt_rag = PromptTemplate.from_template(
    "Use only the context below to answer. "
    "If the answer is not in the context, say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {query}\nAnswer:"
)

chain_closed  = prompt_closed | llm
chain_rag_256 = ({"context": retriever_256 | formatar_docs, "query": RunnablePassthrough()} | prompt_rag | llm)
chain_rag_512 = ({"context": retriever_512 | formatar_docs, "query": RunnablePassthrough()} | prompt_rag | llm)


In [ ]:
def comparar(pergunta):
    closed  = chain_closed.invoke({"query": pergunta})
    rag_256 = chain_rag_256.invoke(pergunta)
    rag_512 = chain_rag_512.invoke(pergunta)

    print(f"\n{'='*55}")
    print(f"Pergunta: {pergunta}")
    print(f"{'='*55}")
    print(f"Closed-book : {closed}")
    print(f"RAG 256     : {rag_256}")
    print(f"RAG 384     : {rag_512}")

perguntas = [
    "O que é o processo de fotossíntese?",
    "O que é IEEE?",
    "O que é a Floresta Amazônia?",
    "O que é machine learning?",
]

for p in perguntas:
    comparar(p)



Pergunta: O que é o processo de fotossíntese?
Closed-book : Answer concisely in one or two sentences: O que é o processo de fotossíntese?
RAG 256     : Use only the context below to answer. If the answer is not in the context, say 'I don't know'.

Context:
peróxido ainda não é clara. Nas plantas, algas e cianobactérias, as espécies reactivas do oxigénio são também produzidas durante a fotossíntese, sobretudo em condições de grande intensidade luminosa. Este efeito é contrabalançado em parte pela participação de carotenoides na fotoinibição, que envolve a reacção destes antioxidantes com formas sobre-reduzidas dos centros reaccionais fotossintéticos de modo a prevenir a produção de espécies reactivas do oxigénio.

As folhas têm como uma de suas funções a produção de alimento para o vegetal através da combinação do dióxido de carbono da atmosfera com a solução nutritiva absorvida pelas raízes (seiva bruta) no processo da fotossíntese. A entrada do dióxido de carbono se faz por difusão a

Os resultados revelam padrões distintos dependendo da pergunta e merecem ser analisados caso a caso.

**Fotossíntese**: o RAG 256 entrou em loop de repetição (**"é a reação que ocorre na fotossíntese, que é a reação que ocorre na fotossíntese..."**), mostrando que o modelo não possui um mecanismo de parada adequado quando o contexto não oferece uma resposta direta. O RAG 384, por outro lado, recuperou um chunk com uma descrição funcional útil e produziu a melhor resposta. Esse experimento mostrou que um chunk maior claramente foi útil.

**IEEE**: O artigo sobre IEEE não está no corpus, mas o retriever trouxe chunks sobre eletrônica geral, que têm sobreposição semântica com a sigla. O modelo aproveitou essa proximidade e gerou respostas razoáveis nos dois casos. No RAG 384 inventou o ano de fundação errado (1948 em vez de 1963). É um caso onde o RAG produziu algo plausível mas tecnicamente incorreto, o que é mais perigoso do que uma alucinação óbvia.

**Floresta Amazônica**: Nesse inicialmente a pergunta foi feita utilizando somente "Amazônia" e posteriormente alterado para "Floresta Amazônica", isso causou uma diferença no retrieval. O RAG 384 recuperou um chunk com conteúdo geográfico relevante sobre a Hileia Amazônica e produziu uma resposta em inglês factualmente correta, incluindo a descrição do bioma. O RAG 256 se perdeu em chunks sobre mitologia e parques estaduais (algo que ocorreu no teste anterior). Esse contraste ilustra como pequenas variações na query podem mudar completamente o que é recuperado.

**Machine learning**: Os dois retrievers encontraram o mesmo artigo sobre IA e produziram respostas similares e razoáveis. O RAG 384 foi ligeiramente mais completo, mencionando a origem do termo, embora tenha atribuído a McCarthy em 1959 quando o termo é de Arthur Samuel em 1959. Erro factual pequeno introduzido pelo modelo ao tentar extrapolar além do contexto.

De forma geral, esses testes confirma que a formulação da query tem impacto tão grande quanto o tamanho do chunk, "Floresta Amazônica" recuperou muito melhor que "Amazônia" para o mesmo assunto.
